# Build a Semantic Cache

The objective  of this Pre-generated Cache is to load Redis VectorDB with frequently asked questions, derived from LLMs (data may also be collected from domain experts or session data of an application). This is often considered as the SAFEST caching strategy because the information can be manually vetted before placing in the cache.

This notebook will use:

* LlamaIndex to parse the pdf file with high fidelity.
* Groq LLM to extract FAQs from the extracted source material.
* Redis to index the FAQs for semantic caching.

### Import Libraries and Connect to resources

In [1]:
import os
import nest_asyncio
from credentials import LLAMA_CLOUD_API_KEY, GROQ_API_KEY, HF_TOKEN

os.environ["LLAMA_CLOUD_API_KEY"] = LLAMA_CLOUD_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# need this for running llama-index code in Jupyter Notebooks
nest_asyncio.apply()

### Initializes LLM client

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.5,
)
print("✓ Groq LLM initialized (Llama-3.1-8B-Instant)")

✓ Groq LLM initialized (Llama-3.1-8B-Instant)


### Initialize Vectorizer

* This text vectorizer is configured with a sentence transformer model from Hugging Face. 
* It encodes text into 384-dimensional float vector embeddings, optimized for semantic similarity tasks.

In [3]:
from redisvl.extensions.cache.llm import SemanticCache
from redisvl.utils.vectorize import HFTextVectorizer

# redisvl requires its own vectorizer class (not LangChain's HuggingFaceEmbeddings)
redisvl_vectorizer = HFTextVectorizer(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Download Dataset

We will use this chevy colorado truck brochure to generate FAQ prompts later

In [4]:
from urllib.request import urlretrieve

# Create data directory
os.makedirs('data', exist_ok=True)

# Download the PDF
url = 'https://raw.githubusercontent.com/redis-developer/LLM-Document-Chat/main/docs/2022-chevrolet-colorado-ebrochure.pdf'
filepath = 'data/2022-chevrolet-colorado-ebrochure.pdf'
urlretrieve(url, filepath)
print(f'Downloaded: {filepath}')

Downloaded: data/2022-chevrolet-colorado-ebrochure.pdf


### Create a Redis VectorDB
* This is used to store and retrieve the FAQ prompt embeddings
* Semantic similarity lookups are performed retrival

In [5]:
import redis
from credentials import REDIS_HOST, REDIS_PORT, REDIS_PASSWORD

# Create Redis client
redis_client = redis.Redis(
  host=REDIS_HOST,
  port=REDIS_PORT,
  password=REDIS_PASSWORD
)

# Test connection
redis_client.ping()


True

In [6]:
# Clear Redis database (optional)
redis_client.flushdb()

True

### Parses the downloaded PDF

We used LlamaCloud SDK to upload and parse a PDF document into a structured Markdown format using an "agentic" workflow. It read the raw PDF file and output an organized Markdown format that code can work with, preserving structure like headers, tables, and lists better than plain text extraction.

# set up LlamaParse agent to convert the PDF into markdown
parser = LlamaParse(
    result_type="markdown"  # Options: "markdown" | "text"
)

file_extractor = {".pdf": parser}
reader = SimpleDirectoryReader("./data", file_extractor=file_extractor)
documents = reader.load_data()

In [12]:
from llama_cloud import LlamaCloud
from llama_index.core import Document

In [11]:
# Initialize LlamaCloud client
client = LlamaCloud(api_key=os.environ["LLAMA_CLOUD_API_KEY"])

# Upload the PDF
file_obj = client.files.create(
    file="./data/2022-chevrolet-colorado-ebrochure.pdf",
    purpose="parse",
)

# Parse the uploaded file and retrieve markdown output
result = client.parsing.parse(
    file_id=file_obj.id,
    tier="agentic",
    version="latest",
    expand=["markdown"],
)

# Convert each parsed page into a llama_index Document
documents = [
    Document(
        text=page.markdown,
        metadata={"page_label": str(i + 1), "file_name": "2022-chevrolet-colorado-ebrochure.pdf"},
    )
    for i, page in enumerate(result.markdown.pages)
]
print(f"✓ Parsed {len(documents)} pages")

✓ Parsed 24 pages


In [16]:
from llama_index.core.node_parser import MarkdownNodeParser

parser = MarkdownNodeParser()

nodes = parser.get_nodes_from_documents(documents)

MarkdownNodeParser is used to split the markdown document into smaller nodes (chunks) for FAQ prompt extraction.

In [17]:
for node in nodes:
  print("############", "\n", node.text)

############ 
 A tan Chevrolet Colorado ZR2 pickup truck is shown driving through a desert landscape, kicking up a large cloud of dust and sand. The vehicle features off-road tires, a blacked-out grille with "CHEVROLET" lettering, and a sports bar with auxiliary lighting in the bed.
############ 
 # 2022 COLORADO
############ 
 # Choose your adventure.

The 2022 Colorado delivers everything you could ask for in a midsize pickup. Engine choices that are powerful and efficient, including an available GM-exclusive Duramax® 2.8L Turbo-Diesel engine that provides up to 7,700 lbs. of towing<sup>1, 2</sup> muscle. A ZR2 off-road beast with the capability to conquer tough trails. And a comfortable interior filled with convenience and technology features. So go ahead. Choose your best life in Colorado.

Colorado Crew Cab ZR2 in Sand Dune Metallic with available ZR2 Dusk Special Edition. Vehicle shown can tow up to 5,000 lbs.<sup>2, 3</sup>

**1** Requires Colorado Crew Cab Short Box LT 2WD with

* The purpose of this cell is to visually inspect how the PDF was split into chunks, so we can verify that the Markdown parsing and chunking worked correctly before passing the nodes to the LLM for FAQprompt extraction. 
* The cell prints a ############ separator followed by the text content of that node.

### Define LLM Output Schema

Pydantic is used to define the exact JSON structure that the LLM must return when generating FAQs prompts:

**Data models (Pydantic):**
- **`PromptResponse`** — represents a single FAQ entry with two fields: `prompt` (the question) and `response` (the answer).
- **`FAQs`** — a wrapper containing a list of `PromptResponse` pairs.

These Pydantic models enforce that the LLM returns structured, typed data rather than free-form text.

**`JsonOutputParser(pydantic_object=FAQs)`:**
- Tells LangChain to parse the LLM's output as JSON and validate it against the `FAQs` schema.
- Auto-generates `format_instructions` (injected into the prompt in the next cell) that tell the LLM exactly what JSON structure to produce.

In [18]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List


# Define your desired data structure.
class PromptResponse(BaseModel):
    prompt: str = Field(description="User input question about information in the document.")
    response: str = Field(description="Grounded answer from the LLM pertaining to the user's question.")

class FAQs(BaseModel):
    pairs: List[PromptResponse] = Field(description="List of prompt response pairs extracted from the document")


# Set up a parser + inject instructions into the prompt template.
json_parser = JsonOutputParser(pydantic_object=FAQs)

### Build the FAQ Generation Prompt & Chain

**`PromptTemplate`** defines the system instructions sent to the LLM for each document chunk (node):
- `{doc}` — replaced at runtime with the text of each node from the PDF.
- `{format_instructions}` — pre-filled by `json_parser.get_format_instructions()`, injecting the exact JSON schema the LLM must follow.

**`faq_generator_chain = prompt | llm | json_parser`** uses LangChain's pipe operator (`|`) to chain three steps:
1. `prompt` — formats the template with the document text
2. `llm` — sends it to the Groq LLM and gets a response
3. `json_parser` — parses and validates the response into the `FAQs` Pydantic structure

When invoked with `faq_generator_chain.invoke({"doc": node.text})`, all three steps run automatically in sequence.

In [19]:
prompt = PromptTemplate(
    template="""You are a document intelligence tool used to extract FAQs
    from portions of a source PDF document. Put yourself in the shoes of a potential
    reader of the provided material and anticipate what questions they might have.
    Looking at the context below, your goal is to pull out as many likely question
    (prompt) and answer (response) pairs as possible, using only what's provided.

    Other Rules:
    - Create as many FAQs as possible, even rewording the same question and answer
    a few times while remaining faithful to the truth and provided content.
    - Ignore heavy marketing or salesly concepts and focus specifically on
    factual data.
    - It's ok if your response is an empty JSON object for a particular section.

    {format_instructions}

    Document Context:\n{doc}\n""",
    input_variables=["doc"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()},
)

faq_generator_chain = prompt | llm | json_parser

In [20]:
def extract_faqs(nodes):
    all_faqs = []
    for i, node in enumerate(nodes):
        print(f"Processing node {i+1} of {len(nodes)}", flush=True)
        results = faq_generator_chain.invoke({"doc": node.text})
        if results and results.get("pairs"):
            all_faqs.extend(results["pairs"])
    return all_faqs

This function iterates over all document nodes and runs the LLM chain on each one to extract FAQs

In [21]:
all_faqs = extract_faqs(nodes)

Processing node 1 of 49
Processing node 2 of 49
Processing node 3 of 49
Processing node 4 of 49
Processing node 5 of 49
Processing node 6 of 49
Processing node 7 of 49
Processing node 8 of 49
Processing node 9 of 49
Processing node 10 of 49
Processing node 11 of 49
Processing node 12 of 49
Processing node 13 of 49
Processing node 14 of 49
Processing node 15 of 49
Processing node 16 of 49
Processing node 17 of 49
Processing node 18 of 49
Processing node 19 of 49
Processing node 20 of 49
Processing node 21 of 49
Processing node 22 of 49
Processing node 23 of 49
Processing node 24 of 49
Processing node 25 of 49
Processing node 26 of 49
Processing node 27 of 49
Processing node 28 of 49
Processing node 29 of 49
Processing node 30 of 49
Processing node 31 of 49
Processing node 32 of 49
Processing node 33 of 49
Processing node 34 of 49
Processing node 35 of 49
Processing node 36 of 49
Processing node 37 of 49
Processing node 38 of 49
Processing node 39 of 49
Processing node 40 of 49
Processin

Calls `extract_faqs` on all document nodes, triggering one LLM API call per node for FAQ extraction. The returned prompt/response pairs are collected into `all_faqs`.

In [22]:
print("Generated", len(all_faqs), "frequently asked questions.")

Generated 295 frequently asked questions.


In [23]:
all_faqs[:10]

[{'prompt': 'What engine options are available in the 2022 Colorado?',
  'response': 'The 2022 Colorado has powerful and efficient engine choices, including an available GM-exclusive Duramax 2.8L Turbo-Diesel engine.'},
 {'prompt': 'What is the towing capacity of the 2022 Colorado?',
  'response': 'The 2022 Colorado has a towing capacity of up to 7,700 lbs. with the Duramax 2.8L Turbo-Diesel engine.'},
 {'prompt': 'What is the towing capacity of the Colorado ZR2?',
  'response': 'The Colorado ZR2 has a towing capacity of up to 5,000 lbs.'},
 {'prompt': 'What is the ZR2 off-road beast capable of?',
  'response': 'The ZR2 off-road beast is capable of conquering tough trails.'},
 {'prompt': 'What features are available in the interior of the 2022 Colorado?',
  'response': 'The 2022 Colorado has a comfortable interior filled with convenience and technology features.'},
 {'prompt': 'What is the Colorado Crew Cab ZR2 shown with?',
  'response': 'The Colorado Crew Cab ZR2 is shown with availa

### Embed Each LLM generated FAQ prompt

This Converts all FAQ prompts into vector embeddings and validates the result, by returning 'True' if exactly one embedding was created per FAQ entry. This ensures nothing is dropped before storing into the Vector DB.

In [24]:
# Embed each FAQ prompt using the redisvl vectorizer
embeddings = redisvl_vectorizer.embed_many([pair["prompt"] for pair in all_faqs])

# Check to make sure we've created enough embeddings, 1 per FAQ record
len(embeddings) == len(all_faqs)


True

### Initialize the Semantic Cache

Initializes `SemanticCache` — the core component that ties everything together:

- **`redis_url`** — Redis connection string in the format `redis://:<password>@<host>:<port>`.
- **`vectorizer`** — uses the same HuggingFace sentence-transformer model to embed incoming query strings at search time, making them comparable to the stored FAQ embeddings.
- **`distance_threshold=0.2`** — maximum vector distance allowed for a cache hit. Lower = stricter matching. Queries farther than `0.2` from any stored entry return a cache miss.

In [25]:
redis_url = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"
cache = SemanticCache(
    vectorizer=redisvl_vectorizer,
    distance_threshold=0.2,
    redis_url=redis_url
)
print("✓ SemanticCache initialized")


✓ SemanticCache initialized


In [26]:
cache.check("testing") # cache should be empty

[]

### Save FAQ prompt, response, and vector embedding into the Semantic Cache

Loads all generated FAQs into Redis by calling `cache.store(...)` for each entry — this is the **indexing** step:

- `if "prompt" in entry and "response" in entry` — guards against malformed entries missing either field.
- `cache.store(prompt, response, vector=embeddings[i])` — stores each FAQ into Redis with:
  - `prompt` — the question string (used as the cache key for similarity matching)
  - `response` — the answer returned on a cache hit
  - `vector` — the pre-computed embedding passed in directly to avoid re-computing it at store time

After this cell runs, Redis contains all FAQ embeddings indexed and ready for semantic similarity searches via `cache.check(...)`.

In [27]:
for i, entry in enumerate(all_faqs):
    if "prompt" in entry and "response" in entry:
        cache.store(prompt=entry["prompt"], response=entry["response"], vector=embeddings[i])


### Query the Semantic Cache

`cache.check(query)` searches Redis Vector DB for a stored FAQ response semantically similar to the input question:

1. The input string is **embedded** into a vector using `redisvl_vectorizer`
2. That vector is compared against all stored FAQ vectors in Redis using vector distance
3. If any stored FAQ prompt has a distance ≤ `0.2`, it returns the matching cached `response`
4. If nothing is close enough, it returns an empty list (cache miss)

Unlike exact-match caching, semantically equivalent questions with different wording will still return a cached answer.

In [ ]:
cache.check("What models of chevy colorado are available?") # cache hit

[{'entry_id': '548756ba36d5561aca1dfad77be050d707cb1ba32d216fc645c9e65361c3a2b1',
  'prompt': 'What are the different models of the Chevrolet Colorado pickup truck?',
  'response': 'The Chevrolet Colorado pickup truck is available in four models: WT, LT, Z71, and ZR2.',
  'vector_distance': 0.112347602844,
  'inserted_at': 1772242134.05,
  'updated_at': 1772242134.05,
  'key': 'llmcache:548756ba36d5561aca1dfad77be050d707cb1ba32d216fc645c9e65361c3a2b1'}]

In [36]:
cache.check("What models of chevy colorado are available?") # cache hit

[{'entry_id': '548756ba36d5561aca1dfad77be050d707cb1ba32d216fc645c9e65361c3a2b1',
  'prompt': 'What are the different models of the Chevrolet Colorado pickup truck?',
  'response': 'The Chevrolet Colorado pickup truck is available in four models: WT, LT, Z71, and ZR2.',
  'vector_distance': 0.112347602844,
  'inserted_at': 1772242134.05,
  'updated_at': 1772242134.05,
  'key': 'llmcache:548756ba36d5561aca1dfad77be050d707cb1ba32d216fc645c9e65361c3a2b1'}]

In [37]:
cache.check("How many miles per galon does the chevy colorado get?") #cache miss

[]

In [30]:
#cache.clear()

In [31]:
#cache.delete()